# 23 - Train FrozenLake DQN with Autoregressive Reasoning

Use extra inference compute on a fixed observed history: the **same Mouse embedding and
backbone** generate vocabulary tokens after the value token. The value head reads the
hidden state at the **consumed end-of-generation token**. Set `END_GENERATION_TOKEN_ID`
to override the stop token, or leave it `None` to use the checkpoint’s generation
defaults (including multiple stop IDs when configured). Only a vocabulary projection and
output norm are added from the same checkpoint; the usual delayed DQN target copy
remains.

Install Mouse Core with its `examples` extra and use a CUDA notebook kernel. Both
uncached training and paged generation use **FlexAttention only**. PyTorch currently
supports CPU FlexAttention inference but not backward. `kernels` is declared in the
`kernels` project extra; `USE_KERNELS` optionally enables compatible Hugging Face
decoder kernels. Use standard Python 3.15; the `all` installation extra includes
`kernels`. Continuous batching recycles rows in Mouse's own paged cache. It does not use
a second generator or Transformers' batching engine.

This is a small executable experiment, not a claim that a few updates solve FrozenLake.
Random replay can contain few successful paths; inspect its reward coverage.

This notebook only loads recorded episodes, generates reasoning, trains, and saves.
Environment collection and evaluation are separate workflows.

## Setup

Imports and configuration. Edit these settings before running the cells below.


In [ ]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory
from typing import cast

import torch
from huggingface_hub import HfApi
from tensordict import TensorDict
from torch import nn
from transformers import AutoTokenizer

from mouse_core import MouseOptimizer
from mouse_core.data import (
    DataLoader,
    EnvironmentProtocol,
    Tokenizer,
    compose,
    environment_decisions,
    load_stores_from_hub,
    save_tokenizer,
)
from mouse_core.models import (
    MLP,
    Backbone,
    BaseBackbone,
    Embedding,
    Model,
    MultiEmbedding,
    save_model,
)
from mouse_core.models.autoregressive_reasoner import (
    AutoregressiveReasoner,
    ReasoningTrace,
)
from mouse_core.models.vocabulary_head import VocabularyHead
from mouse_core.objectives import DqnObjective, boundary_discount, reinforce_loss

# Repositories
DATASET_ID = "mouse-example-frozenlake-reasoning-dataset"
MODEL_ID = "mouse-example-autoregressive-dqn"

# Environment
IS_SLIPPERY = False
MAX_EPISODE_STEPS = 16

# Model
PRETRAINED = "Qwen/Qwen3-0.6B"

# Tokenizer
GENERATION_START_TOKEN = "\n"

# Reasoning
REINFORCE_WEIGHT = 1.0
REASONING_TEMPERATURE = 1.0
MAX_NEW_TOKENS = 128
END_GENERATION_TOKEN_ID: int | None = None
MAX_ACTIVE_REQUESTS = 8

# Training
PREFETCH = 4
SAMPLE_BUDGET = 8
NUM_WORKERS = 0
NUM_CYCLES = 1
TRAIN_STEPS = 10

# Objectives
GAMMA = 0.99

# Optimizer
LR_EMBEDDING = 1e-5
LR_BACKBONE = 1e-5
LR_HEADS = 1e-5
MAX_GRAD_NORM = 1.0

# Target updates
POLYAK_TAU_EMBEDDING = 0.1
POLYAK_TAU_BACKBONE = 0.1
POLYAK_TAU_VALUE = 0.001
# delayed vocabulary head
POLYAK_TAU_REASONING_HEAD = 0.001

# Random seeds
SEED = 7

# Runtime options
USE_KERNELS = False
GRADIENT_CHECKPOINTING = True

## Runtime


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "This notebook trains with FlexAttention and requires CUDA; CPU supports inference only."
    )
device = torch.device("cuda")
torch.manual_seed(SEED)

## Load Data

This is offline RL. Load existing Mouse event stores; this notebook never creates
or steps an environment. Set `DATASET_ID` to a previously recorded dataset of
standalone 4x4 `FrozenLake-v1` episodes, with actions 0=left, 1=down, 2=right, 3=up.
Set `IS_SLIPPERY` and `MAX_EPISODE_STEPS` to match that dataset. The dataset ID is
an example name to replace with your own source. Use the collector's
coordinator/agent/environment event schema with `_episode_start`, `_terminated`,
and `_truncated`; repeated procedural-map tasks require their own matching prompt.

Only complete episodes are sampled. Arrival rows hold the recorded previous
action and reward. Terminal and time-limit rows have no value readout. Generating
reasoning adds model computation over these recorded histories, not interactions.
Evaluate saved checkpoints separately in [15 — Inference](15_inference.ipynb).


In [ ]:
stores = load_stores_from_hub(repo_id=DATASET_ID, split="train")

## Tokenizer

Render each observed arrival as a prompt event. The same tokenizer defines text
inputs, the generation-start marker, and objective labels. The initial event uses
the chat template; later events append the incoming action, reward, and observation.
Terminal observations retain their labels but have no prediction marker.

In [ ]:
def task_instructions() -> str:
    return (
        "Estimate action values in FrozenLake: a 4 by 4 grid, with states numbered row-major 0 to 15. "
        "Actions: 0=left, 1=down, 2=right, 3=up. The map is hidden. "
        "The goal rewards 1; a hole ends the episode with reward 0. "
        f"The episode ends after at most {MAX_EPISODE_STEPS} steps. "
        f"Slippery transitions: {IS_SLIPPERY}. "
        "Think briefly about the observed history, then stop.\nHistory:\n"
    )


def format_observation(*, observation, action, reward, requested):
    """Render one arrival; the tokenizer adds its action-request marker."""
    if action is None:
        text = text_tokenizer.apply_chat_template(
            [
                {
                    "role": "user",
                    "content": task_instructions() + f"State {observation}.\n",
                }
            ],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    else:
        text = f"Action {action}, reward {reward:g}, state {observation}.\n"
    return text + ("Value:" if requested else "")


def format_events(events):
    """Convert sampled source events to prompt events; the loader owns packing."""
    protocol = EnvironmentProtocol(
        episode_start="_episode_start", terminated="_terminated", truncated="_truncated"
    )
    formatted = []
    for row in environment_decisions(events, protocol=protocol):
        observation, action = row["observation"], row.get("action")
        if row.get("inner_terminated") or row.get("inner_truncated"):
            raise ValueError(
                "Use standalone 4x4 FrozenLake episodes, without inner repeats"
            )
        if not 0 <= observation < 16 or (action is not None and not 0 <= action < 4):
            raise ValueError(
                "This example expects 4x4 FrozenLake observations and four actions"
            )
        reward = float(row.get("reward", 0.0))
        terminated = bool(row.get("_terminated", False))
        truncated = bool(row.get("_truncated", False))
        requested = bool(row.get("action_requested", False))
        if not (terminated or truncated) and not requested:
            raise ValueError("Each nonterminal observation must request an action")
        payload = {
            "prompt": format_observation(
                observation=int(observation),
                action=action,
                reward=reward,
                requested=requested,
            ),
            "reward": reward,
            "terminated": terminated,
            "truncated": truncated,
        }
        if action is not None:
            payload["action"] = int(action)
        if requested:
            payload["action_requested"] = True
        formatted.append(
            {
                "provider": "environment",
                "episode_index": row["episode_index"],
                "step_index": row["step_index"],
                "payload": payload,
            }
        )
    return formatted


text_tokenizer = AutoTokenizer.from_pretrained(PRETRAINED)
tokenizer = Tokenizer(
    tokenizers={"text": text_tokenizer},
    step_field="step_index",
    input_fields=[
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "prompt",
            "format": "{prompt}",
        },
        {
            "type": "token",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "action_requested",
            "format": GENERATION_START_TOKEN,
            "head_output": True,
            "autoregressive": True,
        },
    ],
    objective_fields=[
        {
            "input_field": "action",
            "missing_value": None,
        },
        {
            "input_field": "reward",
            "missing_value": 0.0,
        },
        {
            "input_field": "terminated",
            "missing_value": False,
        },
        {
            "input_field": "truncated",
            "missing_value": False,
        },
    ],
)
# Fail before model allocation if the configured marker is not a single token.
tokenizer({"provider": "environment", "payload": {"action_requested": True}})

## Replay

Use the standard `DataLoader` to sample complete recorded episodes. The transform
formats source events and calls `tokenizer.encode`; the loader packs tokens,
objective labels, sequence IDs, and prediction-to-step mappings. `SAMPLE_BUDGET`
fixes the number of complete outer episodes per batch at eight, including all inner
repeats. Generated reasoning has its own cap and does not reduce the sample count.

No environment interactions occur here. Each optimizer update consumes
`loader.next_batch()`, generates fresh reasoning at every action request, then trains.

In [ ]:
train_transform = compose(stages=(format_events, tokenizer.encode))


def sample_ranges(dataset):
    """Keep all events belonging to one outer episode together."""
    start, previous = 0, None
    for index, episode in enumerate(dataset["episode_index"][:]):
        if episode is not None and episode != previous:
            if index:
                yield start, index
            start, previous = index, episode
    if len(dataset):
        yield start, len(dataset)


def sample_eligible(events):
    """Exclude incomplete prefixes and unfinished episode tails."""
    first, last = events[0], events[-1]
    return (
        first["provider"] == "coordinator"
        and first["step_index"] == 0
        and first["payload"].get("_episode_start") is True
        and last["provider"] == "environment"
        and (
            last["payload"].get("_terminated") is True
            or last["payload"].get("_truncated") is True
        )
    )


loader = DataLoader(
    stores=stores,
    sample_budget=SAMPLE_BUDGET,
    sample_ranges=sample_ranges,
    sample_eligible=sample_eligible,
    transform=train_transform,
    prefetch=PREFETCH,
    num_workers=NUM_WORKERS,
    seed=SEED,
)

## Build Encoder and Heads

The same Mouse embedding and backbone generate reasoning tokens and supply the value
head. The frozen target copy supplies DQN backups.


In [ ]:
class ReasoningDqn(Model):
    def __init__(
        self,
        *,
        embedding: MultiEmbedding,
        backbone: BaseBackbone,
        value_head: nn.Module,
        token_head: nn.Module,
        tokenizer_name: str,
    ) -> None:
        super().__init__()
        # Register the supplied objects themselves. No generator model is loaded.
        self.embedding = embedding
        self.backbone = backbone
        self.value_head = value_head
        self.reasoner = AutoregressiveReasoner(token_head)
        self.tokenizer_name = tokenizer_name

    def forward(
        self,
        tokens: TensorDict,
        traces: list[ReasoningTrace],
        *,
        temperature: float,
        score_tokens: bool,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor | None]:
        out = self.reasoner(
            tokens,
            traces,
            embedding=self.embedding,
            backbone=self.backbone,
            tokenizer_name=self.tokenizer_name,
            temperature=temperature,
            score_tokens=score_tokens,
        )
        return (
            self.value_head(out.final_hidden),
            self.value_head(out.value_hidden),
            out.log_prob_sums,
        )


embedding = MultiEmbedding(embeddings={"text": Embedding(pretrained=PRETRAINED)})
backbone = Backbone(
    pretrained=PRETRAINED,
    dtype=torch.float32,
)
backbone.gradient_checkpointing = GRADIENT_CHECKPOINTING
value_head = MLP(
    in_features=backbone.hidden_dim,
    out_features=4,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
    propagate_gradient=1.0,
 rms_norm_eps=backbone.rms_norm_eps)
online = ReasoningDqn(
    embedding=embedding,
    backbone=backbone,
    value_head=value_head,
    token_head=VocabularyHead.from_pretrained(PRETRAINED),
    tokenizer_name="text",
).to(device)
target = online.copy().requires_grad_(False).eval()
if USE_KERNELS:
    backbone.enable_kernels()
    cast(Backbone, target.backbone).enable_kernels()

## Train

For the recorded action, let `y` be the detached DQN backup. Train the final-token value
with ordinary DQN loss. The generated sequence receives reward
`R = -(Q_final(a) - y)^2`. A pre-reasoning baseline uses `b = -(Q_value(a) - y)^2` with
the same target. The extra loss is `-stopgrad(R-b) * sum(log p(token))`, including the
end token, weighted by DQN participation. The normal DQN loss and REINFORCE loss update
the shared model together.

Generation runs without an autograd graph. Recompute each sampled trace to score tokens
and train values. There is no explicit imagination objective. A missing end token raises
before an update; it is not forced or treated as a completed trace. Increase the budget
if the checkpoint fails to stop. Scratchpads are discarded between environment actions.


In [ ]:
optimizer = MouseOptimizer(
    optimizer={"name": "adamw", "betas": (0.9, 0.95), "eps": 1e-8, "weight_decay": 0.0},
    groups={
        "embedding": {
            "online": online.embedding,
            "lr": LR_EMBEDDING,
            "target": target.embedding,
            "polyak_tau": POLYAK_TAU_EMBEDDING,
        },
        "backbone": {
            "online": online.backbone,
            "lr": LR_BACKBONE,
            "target": target.backbone,
            "polyak_tau": POLYAK_TAU_BACKBONE,
        },
        "value_head": {
            "online": online.value_head,
            "lr": LR_HEADS,
            "target": target.value_head,
            "polyak_tau": POLYAK_TAU_VALUE,
        },
        "reasoner.token_head": {
            "online": online.reasoner.token_head,
            "lr": LR_HEADS,
            "target": target.reasoner.token_head,
            "polyak_tau": POLYAK_TAU_REASONING_HEAD,
        },
    },
)

objective = DqnObjective(
    reward=None,
    value=None,
    gate=None,
    temperature=0.0,
    discount=boundary_discount(
        gamma_step=GAMMA, gamma_terminated=0.0, gamma_truncated=0.0
    ),
)


training_step = 0


def run_train(
    *, num_steps: int
) -> tuple[torch.Tensor, dict[str, float | torch.Tensor]]:
    """Sample recorded episodes, generate reasoning, and train one batch per step."""
    global training_step
    if num_steps < 1:
        raise ValueError("num_steps must be positive")
    if not 0 < MAX_GRAD_NORM < float("inf") or not 0 < REINFORCE_WEIGHT < float("inf"):
        raise ValueError("Invalid max_grad_norm or reinforce_weight")
    if (
        objective.value is not None
        or objective.cql_weight != 0
        or objective.gate is not None
    ):
        raise ValueError(
            "This example's token reward is the plain, one-step, untransformed squared DQN error"
        )
    loss: torch.Tensor | None = None
    metrics: dict[str, float | torch.Tensor] = {}
    for _ in range(num_steps):
        tokens, objective_data = loader.next_batch()
        tokens = tokens.to(device)
        objective_data = objective_data.to(device)
        sequence_ids = objective_data["sequence_ids"]
        online.train()
        target.eval()
        traces = online.reasoner.generate(
            tokens,
            embedding=online.embedding,
            backbone=online.backbone,
            tokenizer_name=online.tokenizer_name,
            backend="continuous",
            max_new_tokens=MAX_NEW_TOKENS,
            end_token_id=END_GENERATION_TOKEN_ID,
            max_active_requests=MAX_ACTIVE_REQUESTS,
            do_sample=True,
            temperature=REASONING_TEMPERATURE,
        )
        q, q_before, log_probs = online(
            tokens, traces, temperature=REASONING_TEMPERATURE, score_tokens=True
        )
        assert log_probs is not None
        with torch.no_grad():
            delayed_q, _, _ = target(
                tokens, traces, temperature=REASONING_TEMPERATURE, score_tokens=False
            )
        losses, metrics = objective(
            objective_data=objective_data,
            sequence_ids=sequence_ids,
            step_ids=objective_data["step_ids"],
            predictions=q,
            backup_value_predictions=delayed_q,
            backup_action_predictions=delayed_q,
        )
        td_target = cast(torch.Tensor, metrics["backup"]).detach()
        weights = cast(torch.Tensor, metrics["backup_weight"]).detach()
        step_of = objective_data["step_ids"]
        next_rows = (step_of + 1).clamp_max(len(sequence_ids) - 1)
        actions = torch.where(
            weights > 0, objective_data["action"][next_rows], 0
        ).long()
        selected = q.gather(1, actions[:, None]).squeeze(1)
        selected_before = q_before.gather(1, actions[:, None]).squeeze(1)
        rewards = -(selected - td_target).square().detach()
        baseline = -(selected_before - td_target).square().detach()
        token_loss = reinforce_loss(
            log_prob_sums=log_probs, rewards=rewards, baseline=baseline, weights=weights
        )
        loss = losses["action_value"] + REINFORCE_WEIGHT * token_loss
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        norm = nn.utils.clip_grad_norm_(
            online.parameters(), MAX_GRAD_NORM, error_if_nonfinite=True
        )
        optimizer.step(training_step)
        training_step += 1
        metrics = {
            "dqn_loss": float(losses["action_value"].detach()),
            "reinforce_loss": float(token_loss.detach()),
            "mean_advantage": float(
                ((rewards - baseline) * weights).sum() / weights.sum().clamp_min(1)
            ),
            "grad_norm": float(norm),
            "mean_generated_tokens": sum(len(trace.generated_ids) for trace in traces)
            / len(traces),
        }
    assert loss is not None
    return (loss.detach(), metrics)

## Run

Train on the fixed replay with fresh sampled reasoning at each update.


In [ ]:
try:
    for cycle in range(NUM_CYCLES):
        loss, metrics = run_train(num_steps=TRAIN_STEPS)
        print(f"cycle={cycle} train  loss={loss.item():.4f}  {metrics}")
finally:
    loader.close()

## Save to Hugging Face

Run this cell to upload the online weights, tokenizer files and inference settings
together to `MODEL_ID` in your authenticated Hugging Face account. Temporary files
are only used to assemble the upload. Optimizer and delayed-network state are not
included. Evaluate the uploaded repository in the matching reasoning section of
[15 — Inference](15_inference.ipynb).


In [ ]:
with TemporaryDirectory() as directory:
    output = Path(directory)
    save_model(model=online, path=output)
    save_tokenizer(tokenizer=tokenizer, path=output / "tokenizer")
    text_tokenizer.save_pretrained(output / "text_tokenizer")
    (output / "reasoning_settings.json").write_text(
        json.dumps(
            {
                "reasoning_kind": "autoregressive",
                "pretrained": PRETRAINED,
                "environment": {
                    "id": "FrozenLake-v1",
                    "map_name": "4x4",
                    "is_slippery": IS_SLIPPERY,
                    "max_episode_steps": MAX_EPISODE_STEPS,
                },
                "prompt_prefix": task_instructions(),
                "num_actions": 4,
                "end_token_id": END_GENERATION_TOKEN_ID,
                "max_new_tokens": MAX_NEW_TOKENS,
                "temperature": REASONING_TEMPERATURE,
            },
            indent=2,
        )
        + "\n"
    )
    api = HfApi()
    repository = api.create_repo(
        repo_id=MODEL_ID, repo_type="model", private=False, exist_ok=True
    )
    api.upload_folder(
        repo_id=repository.repo_id,
        repo_type="model",
        folder_path=directory,
        commit_message="Upload reasoning model, tokenizers and inference settings",
    )
print(f"Uploaded checkpoint bundle to {repository}")